# **Tutorial 1 SpatialEx Translates Histology to Omics at Single-Cell Resolution**

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('/data/user/hesy/projects/mini_project/change_data')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
sys.modules.pop('SpatialEx', None)
import SpatialEx as se
print(se.__file__)
device = 'cuda:0'

/data/user/hesy/miniconda3/envs/spatialex/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/data/user/hesy/projects/mini_project/change_data/SpatialEx/__init__.py


## 1. Prepare the dataset

### Choice 1: [Download](https://www.10xgenomics.com/products/xenium-in-situ/human-breast-dataset-explorer) the Xenium Human Breast Cancer tissue dataset.

We provide H&E patch representations generated by different H&E foundation models in the following links, and the expression data is preprocessed:

**UNI (Default)**: [Slice 1](https://drive.google.com/file/d/1730OXeBG6TDQ6ejs5oRGKYhdNXbIU19i/view?usp=sharing) and [Slice 2](https://drive.google.com/file/d/17WhaKtG3iXuZuubIJEi4Y0_0z1TMKIRx/view?usp=sharing)


**CONCH**: [Slice 1](https://drive.google.com/file/d/1WmSN4EWkmj1VFKZZbwWl3oGZAa50NMRN/view?usp=drive_link) and [Slice 2](https://drive.google.com/file/d/1hrhxLl7gBVZykiqxjLc0Od9NAbzb_t_T/view?usp=drive_link)

**Gigapath**: [Slice 1](https://drive.google.com/file/d/1x-ZYl7Nda53_VXCqpHO0OloC5g2DPS0-/view?usp=drive_link) and [Slice 2](https://drive.google.com/file/d/16aypi0NFTDsYr6Yuh3vf1b7SbiqehX6c/view?usp=drive_link)

**Phikon**: [Slice 1](https://drive.google.com/file/d/1jiwdqXjQp240jN0F3XZhVW8JJZ1YQIoR/view?usp=drive_link) and [Slice 2](https://drive.google.com/file/d/1qF3XQR0C0PmYQnzeyKu68K7sngA_y2L-/view?usp=drive_link)

**ResNet50**: [Slice 1](https://drive.google.com/file/d/1aamspsRvPPcKIyApqieAspEqwFcvxgea/view?usp=drive_link) and [Slice 2](https://drive.google.com/file/d/1DwkotYbpeJKWlGFEM6enH2LC3Q4VVeRb/view?usp=drive_link)

### Choice 2: Preprocess your own data from scratch.
```
datasets/
│
├── Human_Breast_Cancer_Rep1/                  # The 1st slice
│   ├── cell_feature_matrix.h5                    
│   ├── cells.csv
│   ├── Xenium_FFPE_Human_Breast_Cancer_Rep1_he_image.ome.tif          
│   ├── Xenium_FFPE_Human_Breast_Cancer_Rep1_he_imagealignment.csv
│   ├── HBRC_Rep1_cell_coor.csv               # Cell segmentation result on H&E image
│   ├── HBRC_Rep1_Out_uni.npy
│
├── Human_Breast_Cancer_Rep2/                  # The 2nd slice
│   ├── cell_feature_matrix.h5
│   ├── cells.csv
│   ├── Xenium_FFPE_Human_Breast_Cancer_Rep2_he_image.ome.tif        
│   ├── Xenium_FFPE_Human_Breast_Cancer_Rep2_he_imagealignment.csv
│   ├── HBRC_Rep2_cell_coor.csv
│   ├── HBRC_Rep2_Out_uni.npy
```

### 1.1 Preprocess Slice 1

In [3]:
save_root1 = "/bigdat2/user/hesy/mini_project/datasets/Human_Breast_Cancer_Rep1/"
save_root2 = "/bigdat2/user/hesy/mini_project/datasets/Human_Breast_Cancer_Rep2/"
resolutions = [256,128,64]
image_encoder = 'uni'

In [4]:
file_path1 = save_root1 + 'cell_feature_matrix.h5'
obs_path1 = save_root1 + 'cells.csv'
img_path1 = save_root1 + 'Xenium_FFPE_Human_Breast_Cancer_Rep1_he_image.ome.tif'
transform_mtx_path1 = save_root1 + 'Xenium_FFPE_Human_Breast_Cancer_Rep1_he_imagealignment.csv'
adata1 = se.pp.Read_Xenium(file_path1, obs_path1)
adata1 = se.pp.Preprocess_adata(adata1)                                              

img, scale = se.pp.Read_HE_image(img_path1)
transform_mtx = pd.read_csv(transform_mtx_path1, header=None).values
adata1 = se.pp.Register_physical_to_pixel(adata1, transform_mtx, scale=scale)
he_patches_dict, adata1 = se.pp.Tiling_HE_patches_multiscale(resolutions, adata1, img)
adata1 = se.pp.Extract_HE_patches_representaion_multiscale(he_patches_dict, adata=adata1, image_encoder=image_encoder, device=device, store_key='he')
spot_coor1, spot_count1, adata1 = se.pp.Generate_pseudo_spot(adata1, all_in=True)


======================== Tiling HE patches for each single cells ===========================
max patch radius is  128
Building patches for resolution 256


100%|██████████| 164000/164000 [00:11<00:00, 14033.72it/s]


Building patches for resolution 128


100%|██████████| 164000/164000 [00:07<00:00, 22286.89it/s]


Building patches for resolution 64


...

====================== Extracting HE representations for each cell =========================
The image encoder is uni


100%|██████████| 2563/2563 [11:56<00:00,  3.58it/s]


====================== Extracting HE representations for each cell =========================
The image encoder is uni


100%|██████████| 2563/2563 [11:45<00:00,  3.63it/s]


164000  cells are included in its nearest spot!


In [5]:
spot_coor1, spot_count1, adata1 = se.pp.Generate_pseudo_spot(adata1, all_in=True) 

164000  cells are included in its nearest spot!


In [ ]:
adata1.write_h5ad('/bigdat2/user/hesy/mini_project/datasets/after_process/multifeature_rep1_repeat.h5ad')

### 1.2 Preprocess Slice 2

In [6]:
file_path2 = save_root2 + 'cell_feature_matrix.h5'
obs_path2 = save_root2 + 'cells.csv'
img_path2 = save_root2 + 'Xenium_FFPE_Human_Breast_Cancer_Rep2_he_image.ome.tif'
transform_mtx_path2 = save_root2 + 'Xenium_FFPE_Human_Breast_Cancer_Rep2_he_imagealignment.csv'
adata2 = se.pp.Read_Xenium(file_path2, obs_path2)    
adata2 = se.pp.Preprocess_adata(adata2)

img, scale = se.pp.Read_HE_image(img_path2)
transform_mtx = pd.read_csv(transform_mtx_path2, header=None).values
adata2 = se.pp.Register_physical_to_pixel(adata2, transform_mtx, scale=scale)
he_patches_dict, adata2 = se.pp.Tiling_HE_patches_multiscale(resolutions, adata2, img)
adata2 = se.pp.Extract_HE_patches_representaion_multiscale(he_patches_dict, adata=adata2, image_encoder=image_encoder, store_key='he', device=device)
spot_coor2, spot_count2, adata2 = se.pp.Generate_pseudo_spot(adata2, all_in=True)


======================== Tiling HE patches for each single cells ===========================
max patch radius is  128
Remove the outlier cells: 7025 cells filtered, and Anndata file was reduced!
Building patches for resolution 256


100%|██████████| 111345/111345 [00:09<00:00, 11161.60it/s]


Building patches for resolution 128


100%|██████████| 111345/111345 [00:04<00:00, 24310.33it/s]


Building patches for resolution 64


...

100%|██████████| 1740/1740 [07:49<00:00,  3.71it/s]


====================== Extracting HE representations for each cell =========================
The image encoder is uni


100%|██████████| 1740/1740 [06:38<00:00,  4.36it/s]


====================== Extracting HE representations for each cell =========================
The image encoder is uni


100%|██████████| 1740/1740 [06:31<00:00,  4.45it/s]


In [11]:
spot_coor2, spot_count2, adata2 = se.pp.Generate_pseudo_spot(adata2, all_in=True)

111345  cells are included in its nearest spot!


In [12]:
adata2.write_h5ad('/bigdat2/user/hesy/mini_project/datasets/after_process/multifeature_rep2.h5ad')